Cleaning and conforming only. No aggregation and no row loss.

- `brand_nm` arrives with a leading space on every value (`' LEMON'`).
- `tsr_pckg_nm` has case drift and trailing markers (`20Z NRP 24L`,
  `20z NRP 24L S`, `.591L NRP 24L *`). The raw value is kept as the package
  business key so nothing is destroyed, and the normalized form plus the
  marker are exposed as separate attributes.
- `dollar_volume` is cast to `decimal(18,2)`.
- 260 rows carry a negative dollar volume. They are kept and flagged with
  `is_negative_volume`. The source does not define what they represent, so
  the flag stays technical: they could be returns, credits, reversals or
  corrections, and naming them before the business confirms would put an
  unsupported meaning into the model.

In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'beverage_sales'
SOURCE_SCHEMA_NAME = 'bronze'
SOURCE_TABLE_NAME = 'sales'

TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'silver'
TARGET_TABLE_NAME = 'sales'

In [0]:
df = (
    spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}')
    .select(
        F.to_date(F.col('date'), 'M/d/yyyy').alias('sales_date'),
        F.trim(F.col('ce_brand_flvr')).alias('brand_code'),
        F.upper(F.trim(F.col('brand_nm'))).alias('brand_name'),
        F.upper(F.trim(F.col('btlr_org_lvl_c_desc'))).alias('region'),
        F.upper(F.trim(F.col('chnl_group'))).alias('channel_group'),
        F.upper(F.trim(F.col('trade_chnl_desc'))).alias('trade_channel'),
        F.upper(F.trim(F.col('pkg_cat'))).alias('package_category'),
        F.upper(F.trim(F.col('pkg_cat_desc'))).alias('package_category_desc'),
        F.upper(F.trim(F.col('tsr_pckg_nm'))).alias('package_name'),
        F.col('dollar_volume').cast('decimal(18,2)').alias('dollar_volume'),
        F.col('period').cast('int').alias('period'),
        F.col('source_file'),
        F.col('ingestion_timestamp')
    )
    .withColumn(
        'package_name_std',
        F.trim(F.regexp_replace(F.col('package_name'), r'\s*\*$|\s+S$', ''))
    )
    .withColumn(
        'package_marker',
        F.when(F.col('package_name').rlike(r'\*$'), 'STAR')
        .when(F.col('package_name').rlike(r'\s+S$'), 'S')
        .otherwise('NONE')
    )
    .withColumn('is_negative_volume', F.col('dollar_volume') < 0)
    .filter(F.col('sales_date').isNotNull())
    .filter(F.col('brand_code').isNotNull())
    .filter(F.col('region').isNotNull())
    .filter(F.col('trade_channel').isNotNull())
    .filter(F.col('dollar_volume').isNotNull())
)

In [0]:
df.write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')